# **Length Estimator**

Estimates the output length of every request before generation. Following S³, a DistilBERT classifier reads the request, truncated to its first and last 255 tokens, and predicts one of ten equal-frequency length buckets; the estimate is the mean length of the training answers in the predicted bucket. The classifier is trained on 10,000 Magpie and 10,000 LMSYS-Chat-1M requests that are not in the pool (80% training, 20% validation), chosen on validation by its MAE against the global median and a regression on input length, and tested once on the Magpie requests of the pool. Magpie answers of 3,900 tokens or more are degenerate generations and are removed. The estimate is the length of an answer like those of the training data; the routing pipeline scales it to each model with the verbosity $v_j$.

**Runtime:** T4 GPU. LMSYS-Chat-1M is gated: accept its conditions once on Hugging Face.

## Install

In [1]:
!pip install -q "transformers>=4.44" "datasets>=3.0" accelerate scikit-learn pyarrow

## Configuration

In [ ]:
import json
import os
import time

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sklearn.linear_model import LinearRegression
from transformers import AutoModelForSequenceClassification, AutoTokenizer

SEED = 42
REQUEST_POOL_PATH = "data/request_pool/request_pool_v2.parquet"
OUTPUT_DIR = "data/request_estimates"

# external training set (never in the request pool)
MAGPIE_REPO = "Magpie-Align/Magpie-Llama-3.1-Pro-300K-Filtered"   # same source as the Magpie part of the pool
LMSYS_REPO = "lmsys/lmsys-chat-1m"                                # gated: accept the conditions on its page
N_MAGPIE, N_LMSYS = 10_000, 10_000
VAL_SHARE = 0.20
COUNT_TOKENIZER = "meta-llama/Llama-3.1-8B-Instruct"   # same tokenizer as the request pool (token counts comparable)

# classifier
CLASSIFIER = "distilbert-base-uncased"
N_BUCKETS = 10
MAX_TOKENS_READ = 512        # DistilBERT reads at most 512 tokens: the first 255 and the last 255 of the text
EPOCHS = 3
BATCH_SIZE = 32
LEARNING_RATE = 5e-5
FAILED_GENERATION_TOKENS = 3900   # Magpie answers this long are degenerate (loops, stop mid-sentence): empirical threshold, analysis/length_profile
FAILURE_SOURCE = "magpie"        # the rule applies only to this source (LLM-generated); human references are never removed
EXCLUDE_SOURCES = ["longbench_v1", "longbench_v2"]   # long-context sources excluded from the whole study

rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

Device: cuda


## Token and request pool

In [3]:
try:
    from google.colab import userdata                 # Colab: secret HF_TOKEN
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except ImportError:                                    # local run: environment variable or prompt
    from getpass import getpass
    os.environ["HF_TOKEN"] = os.environ.get("HF_TOKEN") or getpass("Hugging Face token: ")
HF_TOKEN = os.environ["HF_TOKEN"]
os.makedirs(OUTPUT_DIR, exist_ok=True)

pool = pd.read_parquet(REQUEST_POOL_PATH)
pool_texts = set(pool["text"].str.strip())        # whole pool: no training request may come from it
pool_magpie_ids = set(pool.loc[pool["source"] == "magpie", "source_index"].astype(str))
pool = pool[~pool["source"].isin(EXCLUDE_SOURCES)].reset_index(drop=True)
failed_ref = (pool["source"] == FAILURE_SOURCE) & (pool["reference_output_tokens"] >= FAILED_GENERATION_TOKENS)
pool.loc[failed_ref, "reference_output_tokens"] = np.nan
print(f"Excluded sources {EXCLUDE_SOURCES}; pool references treated as generation failures: {int(failed_ref.sum())}")
print(f"Request pool: {len(pool)} requests | with a realistic reference answer: {int(pool['reference_output_tokens'].notna().sum())}")

Mounted at /content/drive
Excluded sources ['longbench_v1', 'longbench_v2']; pool references treated as generation failures: 3
Request pool: 4319 requests | with a realistic reference answer: 2779


## External training set

### Magpie

Requests already in the pool are excluded (same id or same text).

In [4]:
def take(stream, n, keep):
    rows = []
    for ex in stream:
        r = keep(ex)
        if r is not None:
            rows.append(r)
            if len(rows) == n:
                break
    return pd.DataFrame(rows)

def keep_magpie(ex):
    if str(ex.get("uuid")) in pool_magpie_ids or str(ex["instruction"]).strip() in pool_texts:
        return None
    return {"text": ex["instruction"], "answer": ex["response"], "origin": "magpie", "answer_model": "Llama-3.1 (Magpie)"}

magpie_stream = load_dataset(MAGPIE_REPO, split="train", streaming=True, token=HF_TOKEN).shuffle(seed=SEED, buffer_size=20_000)
magpie = take(magpie_stream, N_MAGPIE, keep_magpie)
print(f"Magpie: {len(magpie)} examples")

README.md:   0%|          | 0.00/7.75k [00:00<?, ?B/s]

Magpie: 10000 examples


### LMSYS-Chat-1M

English conversations only; first user message and first answer.

In [5]:
def keep_lmsys(ex):
    conv = ex.get("conversation") or []
    if ex.get("language") != "English" or len(conv) < 2 or conv[0].get("role") != "user" or conv[1].get("role") != "assistant":
        return None
    text, answer = conv[0].get("content", ""), conv[1].get("content", "")
    if not text.strip() or not answer.strip() or text.strip() in pool_texts:
        return None
    return {"text": text, "answer": answer, "origin": "lmsys", "answer_model": ex.get("model")}

lmsys_stream = load_dataset(LMSYS_REPO, split="train", streaming=True, token=HF_TOKEN).shuffle(seed=SEED, buffer_size=20_000)
lmsys = take(lmsys_stream, N_LMSYS, keep_lmsys)
print(f"LMSYS-Chat-1M: {len(lmsys)} examples")

README.md:   0%|          | 0.00/8.88k [00:00<?, ?B/s]

LMSYS-Chat-1M: 10000 examples


### Token counts, duplicates, split

Same tokenizer as the request pool; Magpie answers of `FAILED_GENERATION_TOKENS` or more and duplicated texts are removed; 80% training, 20% validation.

In [6]:
count_tok = AutoTokenizer.from_pretrained(COUNT_TOKENIZER, token=HF_TOKEN)

def count_tokens(texts, batch=256):
    out = []
    for s in range(0, len(texts), batch):
        out += [len(e) for e in count_tok(list(texts[s:s + batch]), add_special_tokens=False)["input_ids"]]
    return np.array(out)

ext = pd.concat([magpie, lmsys], ignore_index=True).drop_duplicates("text").reset_index(drop=True)
ext["input_tokens"] = count_tokens(ext["text"].tolist())
ext["output_tokens"] = count_tokens(ext["answer"].tolist())
failed_ext = (ext["origin"] == FAILURE_SOURCE) & (ext["output_tokens"] >= FAILED_GENERATION_TOKENS)
print(f"Generation failures removed: {int(failed_ext.sum())} ({ext.loc[failed_ext, 'origin'].value_counts().to_dict()})")
ext = ext[~failed_ext & (ext["output_tokens"] > 0)].reset_index(drop=True)
ext["split"] = np.where(rng.random(len(ext)) < VAL_SHARE, "validation", "train")
train, val = ext[ext["split"] == "train"], ext[ext["split"] == "validation"]
print(f"External set: {len(ext)} | train {len(train)} | validation {len(val)}")
print(ext.groupby("origin")["output_tokens"].describe(percentiles=[.1, .5, .9]).round(0).to_string())

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Generation failures removed: 101 ({'magpie': 101})
External set: 19237 | train 15375 | validation 3862
         count   mean    std    min    10%    50%    90%     max
origin                                                          
lmsys   9338.0  189.0  167.0    1.0   16.0  141.0  422.0  2086.0
magpie  9899.0  582.0  266.0  219.0  347.0  538.0  817.0  3826.0


## Length buckets

Ten buckets with the same number of training answers (equal-width buckets, as in S³, would put most answers in the first one); the value of a bucket is the mean length of its answers.

In [7]:
edges = np.unique(np.quantile(train["output_tokens"], np.linspace(0, 1, N_BUCKETS + 1)))
n_buckets = len(edges) - 1

def bucket_of(lengths):
    return np.clip(np.searchsorted(edges, lengths, side="right") - 1, 0, n_buckets - 1)

train = train.assign(bucket=bucket_of(train["output_tokens"].values))
val = val.assign(bucket=bucket_of(val["output_tokens"].values))
bucket_value = train.groupby("bucket")["output_tokens"].mean().reindex(range(n_buckets)).values
print(pd.DataFrame({"from_tokens": edges[:-1], "to_tokens": edges[1:], "value_tokens": bucket_value.round(0),
                    "train_answers": train["bucket"].value_counts().reindex(range(n_buckets)).values}).to_string())

   from_tokens  to_tokens  value_tokens  train_answers
0          1.0       37.0          17.0           1531
1         37.0       98.0          65.0           1534
2         98.0      213.0         151.0           1541
3        213.0      311.0         267.0           1536
4        311.0      383.0         347.0           1541
5        383.0      441.0         413.0           1532
6        441.0      511.0         474.0           1536
7        511.0      602.0         555.0           1542
8        602.0      710.6         651.0           1544
9        710.6     3826.0         957.0           1538


## Simple reference estimators

Global median; regression on input length, $\log \hat{L} = \beta_0 + \beta_1 \log(1 + L^{\mathrm{in}})$.

In [8]:
global_median = float(train["output_tokens"].median())
reg = LinearRegression().fit(np.log1p(train[["input_tokens"]].values), np.log(train["output_tokens"].values))

def predict_median(df):
    return np.full(len(df), global_median)

def predict_regression(df):
    return np.exp(reg.predict(np.log1p(df[["input_tokens"]].values))).clip(1, train["output_tokens"].max())

## DistilBERT bucket classifier (S³)

First 255 and last 255 tokens of the request, plus [CLS] and [SEP].

In [9]:
clf_tok = AutoTokenizer.from_pretrained(CLASSIFIER)
model = AutoModelForSequenceClassification.from_pretrained(CLASSIFIER, num_labels=n_buckets).to(DEVICE)
half = (MAX_TOKENS_READ - 2) // 2

def encode(texts):
    ids = clf_tok(list(texts), add_special_tokens=False, truncation=False)["input_ids"]
    ids = [i if len(i) <= 2 * half else i[:half] + i[-half:] for i in ids]
    ids = [[clf_tok.cls_token_id] + i + [clf_tok.sep_token_id] for i in ids]
    width = max(len(i) for i in ids)
    input_ids = torch.tensor([i + [clf_tok.pad_token_id] * (width - len(i)) for i in ids])
    attention = torch.tensor([[1] * len(i) + [0] * (width - len(i)) for i in ids])
    return input_ids, attention

def batches(df, shuffle):
    order = rng.permutation(len(df)) if shuffle else np.arange(len(df))
    for s in range(0, len(df), BATCH_SIZE):
        yield df.iloc[order[s:s + BATCH_SIZE]]

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
use_amp = DEVICE == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
start = time.time()
for epoch in range(EPOCHS):
    model.train()
    losses = []
    for b in batches(train, shuffle=True):
        input_ids, attention = encode(b["text"])
        labels = torch.tensor(b["bucket"].values)
        with torch.autocast(device_type="cuda", enabled=use_amp):
            loss = model(input_ids=input_ids.to(DEVICE), attention_mask=attention.to(DEVICE), labels=labels.to(DEVICE)).loss
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        losses.append(loss.item())
    print(f"epoch {epoch + 1}: mean loss {np.mean(losses):.3f} | {time.time() - start:.0f} s")

@torch.no_grad()
def predict_bucket(df):
    model.eval()
    out = []
    for b in batches(df, shuffle=False):
        input_ids, attention = encode(b["text"])
        out.append(model(input_ids=input_ids.to(DEVICE), attention_mask=attention.to(DEVICE)).logits.argmax(-1).cpu().numpy())
    return np.concatenate(out)

def predict_classifier(df):
    return bucket_value[predict_bucket(df)]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (560 > 512). Running this sequence through the model will result in indexing errors


epoch 1: mean loss 1.891 | 164 s
epoch 2: mean loss 1.661 | 330 s
epoch 3: mean loss 1.441 | 494 s


## Choice of the estimator (validation set)

MAE, median relative error $|\hat{L} - L| / L$, total-token error $\sum \hat{L} / \sum L - 1$ and bucket accuracy; the estimator with the lowest validation MAE is chosen.

In [10]:
def errors(pred, true):
    return {"MAE_tokens": np.mean(np.abs(pred - true)), "median_rel_error": np.median(np.abs(pred - true) / true),
            "total_token_error": pred.sum() / true.sum() - 1}

ESTIMATORS = {"global median": predict_median, "regression on input length": predict_regression,
              "DistilBERT buckets (S3)": predict_classifier}
y_val = val["output_tokens"].values
val_err = pd.DataFrame({name: errors(f(val), y_val) for name, f in ESTIMATORS.items()}).T
print(val_err.round(3).to_string())
print(f"Bucket accuracy (validation): {np.mean(predict_bucket(val) == val['bucket'].values):.1%}")
print(val.assign(pred=predict_classifier(val)).groupby("origin")
      .apply(lambda g: pd.Series(errors(g["pred"].values, g["output_tokens"].values))).round(3).to_string())
CHOSEN = val_err["MAE_tokens"].idxmin()
print("Chosen estimator:", CHOSEN)

                            MAE_tokens  median_rel_error  total_token_error
global median                  220.663             0.402             -0.033
regression on input length     243.140             0.557             -0.375
DistilBERT buckets (S3)        168.327             0.383              0.113
Bucket accuracy (validation): 29.4%
        MAE_tokens  median_rel_error  total_token_error
origin                                                 
lmsys      112.665             0.553              0.073
magpie     219.402             0.256              0.125
Chosen estimator: DistilBERT buckets (S3)


/tmp/ipykernel_928/2643852702.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series(errors(g["pred"].values, g["output_tokens"].values))).round(3).to_string())


## Test on the request pool (Magpie) and exploratory comparison

The test uses only Magpie, the only pool source whose reference answers are written by an LLM. For Dolly, MATH, TACO and the RouterArena translations, whose references are written by people, the comparison is exploratory and is not an error of the estimator.

In [11]:
test = pool[pool["reference_output_tokens"].notna()].copy()
test["true"] = test["reference_output_tokens"]
is_main = test["source"] == FAILURE_SOURCE

rows = {}
for name, f in ESTIMATORS.items():
    g = test[is_main]
    rows[name] = {"requests": len(g), **errors(f(g), g["true"].values)}
test_err = pd.DataFrame(rows).T
print("TEST (Magpie, LLM-generated references)")
print(test_err.round(3).to_string())

expl = test[~is_main].assign(pred=ESTIMATORS[CHOSEN](test[~is_main]))
exploratory = expl.groupby("source").agg(requests=("true", "size"), median_estimate=("pred", "median"),
                                         median_reference=("true", "median"), sum_estimate=("pred", "sum"),
                                         sum_reference=("true", "sum"))
exploratory["estimate_over_reference"] = exploratory["sum_estimate"] / exploratory["sum_reference"]
print(f"\nEXPLORATORY COMPARISON ({CHOSEN} against references written by people; not an estimator error)")
print(exploratory.drop(columns=["sum_estimate", "sum_reference"]).round(2).to_string())

TEST (Magpie, LLM-generated references)
                            requests  MAE_tokens  median_rel_error  total_token_error
global median                 1929.0     198.957             0.291             -0.318
regression on input length    1929.0     297.409             0.504             -0.527
DistilBERT buckets (S3)       1929.0     179.726             0.231              0.004

EXPLORATORY COMPARISON (DistilBERT buckets (S3) against references written by people; not an estimator error)
             requests  median_estimate  median_reference  estimate_over_reference
source                                                                           
dolly             300           150.62              50.0                     2.55
math              250           956.59             129.5                     4.80
routerarena        50            16.60              53.0                     0.77
taco              250           956.59             209.5                     1.79


## Estimates for the whole pool and save

In [12]:
pool_pred = ESTIMATORS[CHOSEN](pool)
out = pool[["request_id", "source"]].assign(predicted_output_tokens=np.round(pool_pred).astype(int), estimator=CHOSEN)
out_path = os.path.join(OUTPUT_DIR, "request_estimates_v1.parquet")
out.to_parquet(out_path, index=False)

model_dir = os.path.join(OUTPUT_DIR, "length_classifier")
model.save_pretrained(model_dir)
clf_tok.save_pretrained(model_dir)
meta = {"method": "S3 length-bucket classifier (Jin et al., NeurIPS 2023)", "classifier": CLASSIFIER,
        "buckets": "equal-frequency", "bucket_edges": edges.tolist(), "bucket_values": bucket_value.tolist(),
        "max_tokens_read": MAX_TOKENS_READ, "truncation": "first and last tokens", "epochs": EPOCHS,
        "training": {"magpie": int((train["origin"] == "magpie").sum()), "lmsys": int((train["origin"] == "lmsys").sum())},
        "validation_errors": val_err.round(4).to_dict(), "chosen": CHOSEN,
        "test_errors_magpie": test_err.round(4).to_dict(orient="index"),
        "exploratory_human_references": exploratory.round(4).to_dict(orient="index"),
        "count_tokenizer": COUNT_TOKENIZER, "seed": SEED, "excluded_sources": EXCLUDE_SOURCES,
        "failed_generation_tokens": FAILED_GENERATION_TOKENS, "failure_rule_source": FAILURE_SOURCE}
with open(os.path.join(OUTPUT_DIR, "request_estimates_v1_metadata.json"), "w") as f:
    json.dump(meta, f, indent=2, default=str)
print("Saved", out_path, "and the classifier in", model_dir)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved /content/drive/MyDrive/thesis/request_estimates/request_estimates_v1.parquet and the classifier in /content/drive/MyDrive/thesis/request_estimates/length_classifier
